# जीवन प्रत्याशा विश्लेषण

वैश्विक जीवन प्रत्याशा की खोज करने वाले दो डेटासेट:
- **Gapminder** (1952–2007): country, year, population, continent, lifeExp, gdpPercap
- **WHO जीवन प्रत्याशा** (2000–2015): 193 देश, 22 संकेतक (मृत्यु दर, बीएमआई/BMI, जीडीपी, स्कूली शिक्षा, आदि)

यह वर्कबुक **Python** और **R** दोनों में CSV फ़ाइल आयात और डेटा विश्लेषण का प्रदर्शन करती है।

## 1. सेटअप: पैकेज स्थापित करें और डेटासेट डाउनलोड करें

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('pandas + plotly स्थापित किया गया')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"पहले से मौजूद है: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"डाउनलोड किया गया {name}: {lines} पंक्तियाँ")

## 2. Gapminder: Python के साथ अन्वेषण

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"आकार (Shape): {gap.shape}")
print(f"महाद्वीप: {sorted(gap['continent'].unique())}")
print(f"वर्ष सीमा: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# महाद्वीप के अनुसार समय के साथ जीवन प्रत्याशा
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='महाद्वीप के अनुसार जीवन प्रत्याशा (1952-2007)',
              labels={'lifeExp': 'जीवन प्रत्याशा (वर्ष)', 'year': 'वर्ष'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# GDP बनाम जीवन प्रत्याशा (2007), बबल का आकार = जनसंख्या
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='GDP बनाम जीवन प्रत्याशा (2007)',
                 labels={'gdpPercap': 'प्रति व्यक्ति GDP (लॉग)', 'lifeExp': 'जीवन प्रत्याशा'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: R के साथ अन्वेषण

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# महाद्वीप के अनुसार जीवन प्रत्याशा का वितरण (बॉक्सप्लॉट)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "महाद्वीप के अनुसार जीवन प्रत्याशा",
        xlab = "महाद्वीप", ylab = "जीवन प्रत्याशा (वर्ष)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# जीवन प्रत्याशा में सुधार के अनुसार शीर्ष 10 देश (1952 बनाम 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "शीर्ष 10: जीवन प्रत्याशा में वृद्धि (1952-2007)",
        xlab = "बढ़े हुए वर्ष",
        col = "#00CC96", border = NA)

## 4. WHO जीवन प्रत्याशा: Python के साथ अन्वेषण

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"आकार (Shape): {who.shape}")
print(f"कॉलम: {list(who.columns)}")
print(f"\nलापता मान / Missing values (शीर्ष 5):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# विकासशील बनाम विकसित: पूर्व-वर्गीकृत जीवन प्रत्याशा वितरण
# स्पष्ट बार निर्देशांक ब्राउज़र Plotly ब्रिज के माध्यम से लगातार रेंडर होते हैं।
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='जीवन प्रत्याशा: विकासशील बनाम विकसित',
             labels={'Life expectancy': 'जीवन प्रत्याशा (वर्ष)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# स्कूली शिक्षा बनाम जीवन प्रत्याशा
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='स्कूली शिक्षा बनाम जीवन प्रत्याशा (2014)',
                 labels={'Life expectancy': 'जीवन प्रत्याशा (वर्ष)',
                         'Schooling': 'स्कूली शिक्षा के वर्ष'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. WHO जीवन प्रत्याशा: R के साथ अन्वेषण

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nदेश:", length(unique(who$Country)))
cat("\nवर्ष सीमा:", range(who$Year))

In [ ]:
# सहसंबंध: वयस्क मृत्यु दर बनाम जीवन प्रत्याशा
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "वयस्क मृत्यु दर बनाम जीवन प्रत्याशा",
     xlab = "वयस्क मृत्यु दर (प्रति 1000)",
     ylab = "जीवन प्रत्याशा (वर्ष)")
legend("topright", legend = c("विकसित", "विकासशील"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# सरल रैखिक मॉडल: जीवन प्रत्याशा का पूर्वानुमान क्या लगाता है?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## मुख्य निष्कर्ष

- जीवन प्रत्याशा वैश्विक स्तर पर बढ़ी है, लेकिन महाद्वीपों के बीच बड़ा अंतर बना हुआ है
- जीडीपी और स्कूली शिक्षा जीवन प्रत्याशा के मजबूत सकारात्मक भविष्यवक्ता हैं
- वयस्क मृत्यु दर सबसे मजबूत नकारात्मक भविष्यवक्ता है
- विकासशील देश परिणामों में बहुत अधिक भिन्नता (variance) प्रदर्शित करते हैं